In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from scipy.io import loadmat

probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
recording_raw = se.read_intan(f"/home/ubuntu/Downloads/M190011_250519_140739_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)

recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)
recording_f = spre.astype(recording_f, dtype="float32")
recording_f = recording_f.time_slice(start_time = 800, end_time = 5300)
# recording_f = recording_f.remove_channels(remove_channel_ids=['A-002', 'A-004', 'A-008', 'A-009', 'A-010', 'A-018', 
#                                                             'A-020', 'A-022', 'A-023', 'A-024', 'A-026', 'A-028', 'A-030',
#                                                             'A-068', 'A-081', 'A-092', 'A-094', 'A-096', 'A-099', 'A-102',
#                                                             'A-108', 'A-109', 'A-111', 'A-114', 'A-115', 'A-117', 'A-119',
#                                                             'A-120', 'A-123', 'A-125', 'A-126', 'B-000', 'B-002', 'B-005',
#                                                             'B-007', 'B-009', 'B-018', 'B-024', 'B-028', 'B-031', 'B-033',
#                                                             'B-035', 'B-036', 'B-038', 'B-042', 'B-045', 'B-078', 'B-081',
#                                                             'B-085', 'B-087', 'B-096', 'B-105', 'B-117', 'B-121', 'B-122',
#                                                             'B-123', 'B-127'])


read success


In [4]:
output_folder = f'/media/ubuntu/sda/duan/result/250519_filtered_new'
recording_segment_preprocessed = recording_f.save(format="binary", n_jobs=24)

default_params = {
        'detect_sign': 0,  
        'adjacency_radius': 120, 
        'freq_min': 300,  
        'freq_max': 3000,
        'filter': True,
        'whiten': True,  
        'num_workers': 20,
        'clip_size': 50,
        'detect_threshold': 5,
        'detect_interval': 3,  
    }



sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                recording=recording_segment_preprocessed,
                                remove_existing_folder='True',
                                folder=output_folder,
                                **default_params)

# 创建排序分析器
analyzer_mountainsort = si.create_sorting_analyzer(
    sorting=sorting_mountainsort, 
    recording=recording_segment_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

# 计算扩展信息
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

# 读取spikes.npy并检查无效的spike
spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
spikes = np.load(spikes_path)

# 获取recording的总样本数
total_samples = recording_f.get_num_samples()

# 检查第一个和最后一个spike
first_spike_valid = spikes[0]['sample_index'] >= 0
last_spike_valid = spikes[-1]['sample_index'] < total_samples

# 如果第一个或最后一个spike无效，删除所有无效的spike
if not first_spike_valid or not last_spike_valid:
    # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
    valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
    spikes_filtered = spikes[valid_mask]
    
    # 保存过滤后的spikes
    np.save(spikes_path, spikes_filtered)
    print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
    print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
else:
    print("所有spike都在有效范围内")

qm_params = sqm.get_default_qm_params()
analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)

Use cache_folder=/tmp/spikeinterface_cache/tmpxwr3jbt3/03OAO64B
write_binary_recording 
engine=process - n_jobs=24 - samples_per_chunk=10,000 - chunk_memory=9.77 MiB - total_memory=234.38 MiB - chunk_duration=1.00s


write_binary_recording (workers: 24 processes): 100%|██████████| 4500/4500 [05:09<00:00, 14.55it/s]


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


estimate_sparsity (no parallelization): 100%|██████████| 4500/4500 [00:13<00:00, 322.39it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 135.65it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2701: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
Compute : spike_locations (workers: 20 processes): 100%|██████████| 4500/4500 [00:01<00:00, 2557.72it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/quality_metric_calculator.p

所有spike都在有效范围内


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1505: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  inds_confidence90 = np.row_stack(np.where(conf_matrix[:, test_rp_centers_mask] > 0.9))
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1059: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
extract PCs (workers: 20 processes): 100%|██████████| 4500/4500 [00:35<00:00, 126.58it/s]


Run:
phy template-gui  /media/ubuntu/sda/duan/result/250519_filtered_new/phy_folder_for_kilosort/params.py


In [29]:
output_dir = '/media/ubuntu/sda/duan/result/250519_filtered_new'
recording_segment_preprocessed = recording_f.save(format="binary", n_jobs=30)

sampling_frequency = recording_segment_preprocessed.get_sampling_frequency()
phy_folder = f'{output_dir}/phy_folder_for_kilosort'

print("读取统一的sorting结果...")
sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise"])
print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")

print("创建analyzer并计算extensions...")
analyzer_curated_phy = si.create_sorting_analyzer(
    sorting=sorting_curated_phy, 
    recording=recording_segment_preprocessed,  # 使用common reference后的recording
    format='binary_folder',
    folder=output_dir + '/analyzer_curated_temp',
    n_jobs=20, verbose = False
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "templates",
    "unit_locations",
    "template_similarity"
]

extension_params = {
    "random_spikes": {"method": "all"},
    "unit_locations": {"method": "center_of_mass"},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
print("完成extensions计算\n")

# 获取neuron信息（整个recording）
templates_ext = analyzer_curated_phy.get_extension("templates")

Use cache_folder=/tmp/spikeinterface_cache/tmp8979vya6/6RTIX4KP
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=9.77 MiB - total_memory=292.97 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 4500/4500 [05:42<00:00, 13.16it/s]


读取统一的sorting结果...
读取到 125 个units

创建analyzer并计算extensions...


compute_waveforms (workers: 20 processes): 100%|██████████| 4500/4500 [01:07<00:00, 66.27it/s] 


完成extensions计算



In [46]:
output_dir = '/media/ubuntu/sda/duan/result/250519_filtered_new'

sampling_frequency = recording_f.get_sampling_frequency()
phy_folder = f'{output_dir}/phy_folder_for_kilosort'

cluster_group_df = pd.read_csv(f'{phy_folder}/cluster_group.tsv', sep='\t')
cluster_info_df = pd.read_csv(f'{phy_folder}/cluster_info.tsv', sep='\t')
template_ind = np.load(f'{phy_folder}/template_ind.npy')
templates = np.load(f'{phy_folder}/templates.npy')
spike_times_npy = np.load(f'{phy_folder}/spike_times.npy')
spike_clusters_npy = np.load(f'{phy_folder}/spike_clusters.npy')
channel_positions = np.load(f'{phy_folder}/channel_positions.npy')

good_mask = cluster_group_df['group'] == 'good'
good_cluster_ids = cluster_group_df.loc[good_mask, 'cluster_id'].values
template_ind_good = template_ind[good_cluster_ids]
templates_good = templates[good_cluster_ids]
n_good = len(good_cluster_ids)
print(f"从phy读取并筛选 good units: {n_good} 个")

templates_abs = np.abs(templates_good)
peak_t = np.argmax(np.sum(templates_abs, axis=2), axis=1)
weights_peak = templates_abs[np.arange(n_good), peak_t, :]
weights_peak = weights_peak / (np.sum(weights_peak, axis=1, keepdims=True) + 1e-10)

unit_locations = np.zeros((n_good, 2))
for i in range(n_good):
    ch_indices = template_ind_good[i]
    pos = channel_positions[ch_indices]
    unit_locations[i] = np.dot(weights_peak[i], pos)

weighted_waveforms = np.array([np.dot(templates_good[i], weights_peak[i]) for i in range(n_good)])
norms = np.linalg.norm(weighted_waveforms, axis=1, keepdims=True)
norms[norms < 1e-10] = 1.0
weighted_norm = weighted_waveforms / norms
template_similarity = np.dot(weighted_norm, weighted_norm.T)

distance_threshold = 10.0
similarity_threshold = 0.95
unit_distances = scipy.spatial.distance.cdist(unit_locations, unit_locations, metric="euclidean")
pair_mask = np.zeros((n_good, n_good), dtype=bool)
for i in range(n_good):
    for j in range(i + 1, n_good):
        if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
            pair_mask[i, j] = True
            pair_mask[j, i] = True

n_components, labels = connected_components(csgraph=pair_mask, directed=False, return_labels=True)

merge_unit_groups = []
for component_id in range(n_components):
    unit_indices = np.where(labels == component_id)[0]
    if len(unit_indices) > 1:
        merge_unit_groups.append(unit_indices.tolist())

cluster_id_to_final_unit_id = {}
final_unit_rep_good_idx = []
final_unit_locations_list = []
good_idx_to_final = np.full(n_good, -1)
final_id = 0
for component_id in range(n_components):
    unit_indices = np.where(labels == component_id)[0]
    rep_idx = unit_indices[0]
    for idx in unit_indices:
        cid = good_cluster_ids[idx]
        cluster_id_to_final_unit_id[cid] = final_id
        good_idx_to_final[idx] = final_id
    final_unit_rep_good_idx.append(rep_idx)
    final_unit_locations_list.append(unit_locations[rep_idx])
    final_id += 1

n_final = final_id
unit_ids_list_final = list(range(n_final))
unit_locations_final = np.array(final_unit_locations_list)

channel_ids_list = list(recording_f.get_channel_ids())

position_waveforms_final = []
extremum_channels_final = {}
for final_unit_id in range(n_final):
    rep_idx = final_unit_rep_good_idx[final_unit_id]
    tmpl = templates_good[rep_idx]
    ch_indices = template_ind_good[rep_idx]
    w = weights_peak[rep_idx]
    position_waveform = np.dot(tmpl, w)
    position_waveforms_final.append(position_waveform)
    peak_flat = np.argmin(tmpl)
    peak_ch = peak_flat % 6
    phy_ch = ch_indices[peak_ch]
    extremum_channels_final[final_unit_id] = str(channel_ids_list[phy_ch])

position_waveforms_final = np.array(position_waveforms_final)
if len(merge_unit_groups) > 0:
    print(f"发现 {len(merge_unit_groups)} 组需要merge的units，合并后共 {n_final} 个units\n")
else:
    print("无需merge units\n")

print("计算每个unit的channel_id...")
channel_ids_dict = {}
for final_unit_id in unit_ids_list_final:
    rep_idx = final_unit_rep_good_idx[final_unit_id]
    ch_indices = template_ind_good[rep_idx]
    channel_ids_dict[final_unit_id] = [str(channel_ids_list[c]) for c in ch_indices]

print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

good_cluster_set = set(good_cluster_ids)
if spike_times_npy.dtype == np.float32 or spike_times_npy.dtype == np.float64:
    spike_sample_indices = (spike_times_npy.flatten() * sampling_frequency).astype(np.int64)
else:
    spike_sample_indices = spike_times_npy.flatten().astype(np.int64)
spike_clusters_flat = spike_clusters_npy.flatten()

neuron_inf_all = {}
for idx, unit_id in enumerate(unit_ids_list_final):
    neuron_inf_all[unit_id] = {
        'location_x': float(unit_locations_final[idx, 0]),
        'location_y': float(unit_locations_final[idx, 1]),
        'position_waveform': position_waveforms_final[idx],
        'extremum_channel': extremum_channels_final[unit_id],
        'channel_id': channel_ids_dict[unit_id]
    }

print("生成整体的gt_detect_array...")
gt_detect_data_all = []
for t, c in zip(spike_sample_indices, spike_clusters_flat):
    if c not in cluster_id_to_final_unit_id:
        continue
    unit_id = cluster_id_to_final_unit_id[c]
    gt_detect_data_all.append({
        'time': int(t),
        'unit_id': unit_id,
        'extremum_channel': extremum_channels_final[unit_id],
    })

gt_detect_array_all = pd.DataFrame(gt_detect_data_all)
print(f"完成gt_detect_array生成，共{len(gt_detect_array_all)}个spikes\n")

print("保存整体的neuron_inf_all和gt_detect_array_all...")
with open(output_dir + '/neuron_inf_all.pickle', 'wb') as f:
    pickle.dump(neuron_inf_all, f)
gt_detect_array_all.to_csv(output_dir + '/gt_detect_array_all.csv', index=False)

从phy读取并筛选 good units: 125 个
发现 2 组需要merge的units，合并后共 123 个units

计算每个unit的channel_id...
完成channel_id计算，共处理123个units

生成整体的gt_detect_array...
完成gt_detect_array生成，共1131339个spikes

保存整体的neuron_inf_all和gt_detect_array_all...


In [70]:
output_dir = '/media/ubuntu/sda/duan/result/250519_filtered_new'

with open(output_dir + '/neuron_inf_all.pickle', 'rb') as f:
    neuron_inf = pickle.load(f)

gt_detect_array = pd.read_csv(output_dir + '/gt_detect_array_all.csv')
gt_detect_array['time'] = gt_detect_array['time'] + 800 * 10000

In [76]:
rec_params = pd.read_csv("/media/ubuntu/sda/duan/result/250519/rec_params.csv")

rec_params = rec_params[rec_params['bhv_codes'] == 10]
rec_params.index = range(len(rec_params))
rec_params = rec_params[(rec_params['bhv_codes_times'] >= 800) & (rec_params['bhv_codes_times'] < 5300)]
# rec_params['bhv_codes_times'] = rec_params['bhv_codes_times'] - 800
rec_params['rec_codes_points'] = rec_params['rec_codes_points'] /3
rec_params = rec_params[rec_params['trial_error'] == 0.0]


In [78]:
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np

output_dir = '/media/ubuntu/sda/duan/result/250519_filtered_new'

SAMPLING_RATE = 10000
sample_to_ms = 1000.0 / SAMPLING_RATE
pre_onset_ms = 50
post_onset_ms = 400
n_time_bins = pre_onset_ms + post_onset_ms

all_neuron_ids = sorted(gt_detect_array['unit_id'].unique())
trial_onsets_samples = rec_params['rec_codes_points'].astype(np.int64).values

n_trials = len(trial_onsets_samples)
n_units = len(all_neuron_ids)


def compute_raster_for_neuron(neuron_id, neuron_idx, gt_detect_array, trial_onsets_samples,
                              pre_onset_ms, post_onset_ms, sample_to_ms, n_time_bins):
    neuron_spikes = gt_detect_array.loc[gt_detect_array['unit_id'] == neuron_id, 'time'].values.astype(np.int64)
    neuron_spikes.sort()
    neuron_raster = np.zeros((n_trials, n_time_bins), dtype=np.float32)
    if neuron_spikes.size == 0:
        return neuron_idx, neuron_raster
    pre_samples = int(pre_onset_ms / sample_to_ms)
    post_samples = int(post_onset_ms / sample_to_ms)
    for trial_idx, onset in enumerate(trial_onsets_samples):
        start_samples = onset - pre_samples
        end_samples = onset + post_samples
        trial_spikes = neuron_spikes[(neuron_spikes >= start_samples) & (neuron_spikes < end_samples)]
        if trial_spikes.size == 0:
            continue
        rel_ms = (trial_spikes - onset) * sample_to_ms
        for t_ms in rel_ms:
            bin_idx = int(np.floor(t_ms + pre_onset_ms))
            if 0 <= bin_idx < n_time_bins:
                neuron_raster[trial_idx, bin_idx] += 1
    return neuron_idx, neuron_raster

print("Computing raster matrix (MATLAB-like logic)...")
raster_results = Parallel(n_jobs=1)(
    delayed(compute_raster_for_neuron)(
        neuron_id,
        neuron_idx,
        gt_detect_array,
        trial_onsets_samples,
        pre_onset_ms,
        post_onset_ms,
        sample_to_ms,
        n_time_bins,
    )
    for neuron_idx, neuron_id in enumerate(tqdm(all_neuron_ids, desc="Raster computation"))
)

raster_unit_trial_time = np.zeros((n_units, n_trials, n_time_bins), dtype=np.float32)
for neuron_idx, neuron_raster in raster_results:
    raster_unit_trial_time[neuron_idx] = neuron_raster

print("Raster shape (units, trials, time):", raster_unit_trial_time.shape)


Computing raster matrix (MATLAB-like logic)...


Raster computation: 100%|██████████| 123/123 [00:01<00:00, 85.42it/s]


Raster shape (units, trials, time): (123, 1039, 450)


In [128]:
import numpy as np
import pickle

stimulus_ids = rec_params['trial_condition'].astype(str).values
stimulus_unique = np.unique(stimulus_ids)
stimulus_id_to_index = {sid: idx for idx, sid in enumerate(stimulus_unique)}
img_idx = np.array([stimulus_id_to_index[s] for s in stimulus_ids], dtype=np.int64)

boot_times = 5

def give_me_nc_py(raster_1d, img_idx, interested_imgs, boot_times):
    r_vals = np.zeros(boot_times, dtype=float)
    for b in range(boot_times):
        d1 = []
        d2 = []
        for img in interested_imgs:
            loc = np.where(img_idx == img)[0]
            n = loc.size
            if n < 2:
                continue
            half = n // 2
            order = np.random.permutation(n)
            first = loc[order[:half]]
            second = loc[order[half:]]
            if first.size == 0 or second.size == 0:
                continue
            d1.append(raster_1d[first].mean())
            d2.append(raster_1d[second].mean())
        if len(d1) == 0 or len(d2) == 0:
            r_vals[b] = np.nan
        else:
            r_vals[b] = np.corrcoef(np.array(d1), np.array(d2))[0, 1]
    r = np.nanmean(r_vals)
    if np.isnan(r):
        return np.nan
    return (2.0 * r) / (1.0 + r)

n_units, n_trials, n_time = raster_unit_trial_time.shape
basic_time_bin_start = 70 + pre_onset_ms
basic_time_bin_end = 220 + pre_onset_ms
basic_slice = slice(int(basic_time_bin_start), int(basic_time_bin_end) + 1)

bin_1 = np.arange(20, 201, 10)
bin_2 = np.arange(90, 391, 10)
window_summary = []
for t1 in bin_1:
    for t2 in bin_2:
        if t1 < t2:
            window_summary.append((t1, t2))
        else:
            window_summary.append((t2, t1))
window_summary = np.array(window_summary, dtype=int)
bin_size = window_summary.shape[0]

unique_imgs = np.unique(img_idx)
if unique_imgs.size >= 1000:
    imgs_for_perm = unique_imgs[:1000]
else:
    imgs_for_perm = unique_imgs
order_imgs = np.random.permutation(imgs_for_perm.size)
half_imgs = imgs_for_perm.size // 2
find_sample = imgs_for_perm[order_imgs[:half_imgs]]
test_sample = imgs_for_perm[order_imgs[half_imgs:]]

from joblib import Parallel, delayed

reliability_basic = np.zeros(n_units, dtype=float)
reliability_best = np.zeros(n_units, dtype=float)
best_r_time1 = np.zeros(n_units, dtype=float)
best_r_time2 = np.zeros(n_units, dtype=float)


def compute_reliability_for_unit(unit_idx):
    raster_this = raster_unit_trial_time[unit_idx]
    total_spikes_per_trial = raster_this.sum(axis=1)
    valid_trial_mask = total_spikes_per_trial >= 2.0
    if not np.any(valid_trial_mask):
        return (
            unit_idx,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
        )
    raster_valid = raster_this[valid_trial_mask]
    img_idx_valid = img_idx[valid_trial_mask]
    basic_raster = raster_valid[:, basic_slice].mean(axis=1)
    rb = give_me_nc_py(basic_raster, img_idx_valid, test_sample, boot_times)
    single_r_pool = np.zeros(bin_size, dtype=float)
    for i in range(bin_size):
        t1 = window_summary[i, 0]
        t2 = window_summary[i, 1]
        s = slice(int(t1 + pre_onset_ms), int(t2 + pre_onset_ms) + 1)
        selected_raster = raster_valid[:, s].mean(axis=1)
        single_r_pool[i] = give_me_nc_py(selected_raster, img_idx_valid, find_sample, boot_times)
    if np.all(np.isnan(single_r_pool)):
        return (
            unit_idx,
            rb,
            np.nan,
            np.nan,
            np.nan,
        )
    best_idx = np.nanargmax(single_r_pool)
    t1_best = window_summary[best_idx, 0]
    t2_best = window_summary[best_idx, 1]
    s_best = slice(int(t1_best + pre_onset_ms), int(t2_best + pre_onset_ms) + 1)
    selected_raster_best = raster_valid[:, s_best].mean(axis=1)
    rbest = give_me_nc_py(selected_raster_best, img_idx_valid, imgs_for_perm, boot_times)
    return (
        unit_idx,
        rb,
        rbest,
        float(t1_best),
        float(t2_best),
    )

results = Parallel(n_jobs=1)(
    delayed(compute_reliability_for_unit)(unit_idx) for unit_idx in range(n_units)
)

for unit_idx, rb, rbest, t1_best, t2_best in results:
    reliability_basic[unit_idx] = rb
    reliability_best[unit_idx] = rbest
    best_r_time1[unit_idx] = t1_best
    best_r_time2[unit_idx] = t2_best

for idx, unit_id in enumerate(all_neuron_ids):
    if unit_id in neuron_inf:
        neuron_inf[unit_id]['reliability_basic'] = float(reliability_basic[idx])
        neuron_inf[unit_id]['reliability_best'] = float(reliability_best[idx])
        neuron_inf[unit_id]['best_r_time1'] = float(best_r_time1[idx])
        neuron_inf[unit_id]['best_r_time2'] = float(best_r_time2[idx])

with open(output_dir + '/neuron_inf_with_reliability.pickle', 'wb') as f:
    pickle.dump(neuron_inf, f)


In [129]:
psth_window_size_ms = 20
half_win = psth_window_size_ms // 2
n_units, n_trials, n_time_bins = raster_unit_trial_time.shape

firing_rate_unit_trial_time = np.zeros((n_units, n_trials, n_time_bins), dtype=np.float32)

for t in range(n_time_bins):
    if t < half_win:
        start, end = 0, psth_window_size_ms
    elif t + half_win >= n_time_bins:
        start, end = n_time_bins - psth_window_size_ms, n_time_bins
    else:
        start, end = t - half_win, t + half_win
    window_len = end - start
    firing_rate_unit_trial_time[:, :, t] = (
        raster_unit_trial_time[:, :, start:end].sum(axis=2) * 1000.0 / window_len
    )

print("Firing rate shape (units, trials, time_bins):", firing_rate_unit_trial_time.shape)
print("Firing rate unit: Hz (same as MATLAB response_matrix_img per trial)")

Firing rate shape (units, trials, time_bins): (123, 1039, 450)
Firing rate unit: Hz (same as MATLAB response_matrix_img per trial)


In [147]:
reliability_threshold = 0.4
mask_reliable = reliability_best >= reliability_threshold
filtered_neuron_ids = [all_neuron_ids[i] for i in range(n_units) if mask_reliable[i]]
n_filtered = len(filtered_neuron_ids)

neuron_inf_filtered = {uid: neuron_inf[uid] for uid in filtered_neuron_ids if uid in neuron_inf}
firing_rate_unit_trial_time_filtered = firing_rate_unit_trial_time[mask_reliable]

with open(output_dir + '/neuron_inf_reliable.pickle', 'wb') as f:
    pickle.dump(neuron_inf_filtered, f)
np.save(output_dir + '/firing_rate_unit_trial_time_reliable.npy', firing_rate_unit_trial_time_filtered)

print(f"reliability_best >= {reliability_threshold}: {n_filtered} units (of {n_units})")
print(f"neuron_inf_filtered saved: {output_dir}/neuron_inf_reliable.pickle")
print(f"firing_rate_unit_trial_time (filtered) shape: {firing_rate_unit_trial_time_filtered.shape}, saved: {output_dir}/firing_rate_unit_trial_time_reliable.npy")

reliability_best >= 0.4: 81 units (of 123)
neuron_inf_filtered saved: /media/ubuntu/sda/duan/result/250519_filtered_new/neuron_inf_reliable.pickle
firing_rate_unit_trial_time (filtered) shape: (81, 1039, 450), saved: /media/ubuntu/sda/duan/result/250519_filtered_new/firing_rate_unit_trial_time_reliable.npy


In [151]:
n_stimuli = len(stimulus_unique)
default_t1, default_t2 = 70, 220

start_time_arr = np.zeros(n_filtered, dtype=np.float64)
end_time_arr = np.zeros(n_filtered, dtype=np.float64)
for idx, uid in enumerate(filtered_neuron_ids):
    if uid in neuron_inf_filtered:
        t1 = neuron_inf_filtered[uid].get('best_r_time1', default_t1)
        t2 = neuron_inf_filtered[uid].get('best_r_time2', default_t2)
        if np.isnan(t1) or np.isnan(t2) or t2 <= t1:
            t1, t2 = default_t1, default_t2
    else:
        t1, t2 = default_t1, default_t2
    start_time_arr[idx] = int(t1 + pre_onset_ms)
    end_time_arr[idx] = int(t2 + pre_onset_ms) + 1

neuron_responses = np.zeros((n_filtered, n_trials), dtype=np.float32)
for i in range(n_filtered):
    start_bin = int(start_time_arr[i])
    end_bin = int(end_time_arr[i])
    if start_bin < 0:
        start_bin = 0
    if end_bin > n_time_bins:
        end_bin = n_time_bins
    if end_bin <= start_bin:
        start_bin = n_time_bins // 2 - 5
        end_bin = n_time_bins // 2 + 5
    neuron_responses[i, :] = firing_rate_unit_trial_time_filtered[i, :, start_bin:end_bin].mean(axis=1)

np.save(output_dir + '/neuron_responses.npy', neuron_responses)
print("neuron_responses shape (n_filtered, n_trials):", neuron_responses.shape)
print("saved:", output_dir + '/neuron_responses.npy')

neuron_responses shape (n_filtered, n_trials): (81, 1039)
saved: /media/ubuntu/sda/duan/result/250519_filtered_new/neuron_responses.npy


In [152]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

n_classes = 24
X = neuron_responses.T
y = img_idx.astype(np.int64)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_classes=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(n_filtered, hidden_dim=128, n_classes=n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X_t = torch.from_numpy(X_train).float().to(device)
y_t = torch.from_numpy(y_train).long().to(device)
train_loader = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)

n_epochs = 50
model.train()
for epoch in range(n_epochs):
    total, correct = 0, 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        correct += (logits.argmax(1) == by).sum().item()
        total += by.size(0)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} train acc: {correct/total:.4f}")

model.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test).float().to(device)
    logits = model(X_test_t)
    test_pred = logits.argmax(1).cpu().numpy()
test_acc = (test_pred == y_test).mean()
print(f"Test accuracy (24 stimuli): {test_acc:.4f} (chance: {1/24:.4f})")

Epoch 10/50 train acc: 0.8075
Epoch 20/50 train acc: 0.9771
Epoch 30/50 train acc: 0.9976
Epoch 40/50 train acc: 0.9976
Epoch 50/50 train acc: 0.9976
Test accuracy (24 stimuli): 0.3558 (chance: 0.0417)
